# Figure Generation and Visualization

This notebook contains the code used to reproduce the main and appendix figures in the paper.

### Organization
1. **Figure 10.** Prediction errors for three coupled dynamical systems at network size $M=10$
2. **Figure 7.** Evaluation metrics for three coupled dynamical systems
3. **Figure 8.** Valid prediction steps for the two methods
4. Utilities for **Figures 2, 6, 11–16, and 18–23**
5. **Figures 2, 6, 11–16, and 18–23.** Weight reconstruction error matrices
6. Utilities for **Figures 5 and 17**
7. **Figure 5.** Reconstructed network topologies
8. **Figure 17.** Reconstructed network topology for the nonlinear product-coupled Lorenz $(x\cdot y)$ system with $M=700$


## Figure 10. Prediction errors for three coupled dynamical systems at network size $M=10$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import os

# ==========================================
# 1. 基础配置
# ==========================================
DIR_TRUE = './generate.data'
DIR_PRED = './result.date'
N_SIZE = 10
TRAIN_LEN = 15000

SYSTEMS = ['llz(x-x)', 'llz(x.y)', 'lsl(x-x)']
METHODS = ['PRC', 'PCI-RC']

# ==========================================
# 2. 数据读取函数 (保持不变)
# ==========================================
def load_and_align_data(sys_name, method, N):
    if 'lsl' in sys_name:
        target_len = 10000
    else:
        target_len = 1000

    y_true_path = ""
    y_pred_path = ""

    if sys_name == 'llz(x-x)':
        y_true_path = os.path.join(DIR_TRUE, 'llz.X-X', f"{N}coupled_lorenz_data.npy")
    elif sys_name == 'llz(x.y)':
        y_true_path = os.path.join(DIR_TRUE, 'llz.xy', f"{N}coupled_lorenz_data.xy.npy")
    elif sys_name == 'lsl(x-x)':
        y_true_path = os.path.join(DIR_TRUE, 'lsl', f"{N}coupled_rossler_data.npy")

    if method == 'PRC':
        fname = ""
        if sys_name == 'llz(x-x)': fname = f"{N}coupled_lorenz_pred.x-x.npy"
        elif sys_name == 'llz(x.y)': fname = f"{N}coupled_lorenz_pred.xy.npy"
        elif sys_name == 'lsl(x-x)': fname = f"{N}coupled_lsl_pred.npy"
        y_pred_path = os.path.join(DIR_PRED, 'PRC', fname)

    elif method == 'PCI-RC':
        sub_folder = ""
        fname = ""
        if sys_name == 'llz(x-x)':
            sub_folder = 'llz.(x-x)'
            fname = f"{N}know_y_pred_traj.(x-x).npy"
        elif sys_name == 'llz(x.y)':
            sub_folder = 'llz.xy'
            fname = f"{N}know_y_pred_traj.(xy).npy"
        elif sys_name =='lsl(x-x)':
            sub_folder = 'lsl'
            fname = f"{N}rossler_know_y_pred_traj.npy"
        y_pred_path = os.path.join(DIR_PRED, sub_folder, 'dynamic', fname)

    try:
        if not os.path.exists(y_true_path): return None, None, target_len
        if not os.path.exists(y_pred_path): return None, None, target_len

        y_true_full = np.load(y_true_path)
        y_pred_full = np.load(y_pred_path)

        if y_true_full.ndim == 2:
            y_true_full = y_true_full.reshape(y_true_full.shape[0], N, -1)
        if y_pred_full.ndim == 2:
            y_pred_full = y_pred_full.reshape(y_pred_full.shape[0], N, -1)

        pred_len = len(y_pred_full)
        use_len = min(pred_len, target_len)
        start_idx = TRAIN_LEN
        end_idx = start_idx + use_len

        if end_idx > len(y_true_full):
            min_l = min(len(y_true_full), len(y_pred_full))
            yt = y_true_full[-min_l:]
            yp = y_pred_full[-min_l:]
            use_len = min_l
        else:
            yt = y_true_full[start_idx : end_idx]
            yp = y_pred_full[:use_len]

        return yt, yp, use_len

    except Exception as e:
        print(f"Error {sys_name} {method}: {e}")
        return None, None, target_len

# ==========================================
# 3. 绘图配置
# ==========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 15,
    'axes.linewidth': 1.2,
    'figure.dpi': 500  # === 修改点：全局设置高清 DPI ===
})
cmap = plt.cm.viridis_r

# === 修改点：在 figure 中也显式指定 dpi=300 ===
fig = plt.figure(figsize=(12, 11), dpi=300)

# 布局设置
outer_grid = gridspec.GridSpec(3, 1, figure=fig,
                               height_ratios=[1, 1, 1],
                               hspace=0.2,
                               left=0.1, right=0.92, top=0.93, bottom=0.08)

group_labels = ['(a)', '(b)', '(c)']

for sys_idx, sys_name in enumerate(SYSTEMS):

    inner_grid = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer_grid[sys_idx],
                                                  width_ratios=[0.97, 0.02], wspace=0.02)

    plot_grid = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=inner_grid[0], hspace=0.1)
    cbar_ax = fig.add_subplot(inner_grid[1])

    data_group = {}
    max_error_in_group = 0.0

    for method in METHODS:
        yt, yp, t_len = load_and_align_data(sys_name, method, N_SIZE)
        if yt is not None and yp is not None:
            diff = yt - yp
            error_mat = np.linalg.norm(diff, axis=2).T
            current_max = np.max(error_mat)
            if current_max > max_error_in_group:
                max_error_in_group = current_max
            data_group[method] = {'error': error_mat, 't_len': t_len, 'found': True}
        else:
            data_group[method] = {'found': False, 't_len': 1000 if 'lsl' not in sys_name else 10000}

    if max_error_in_group < 1e-6: max_error_in_group = 1.0

    # --- 绘图 ---
    last_im = None
    for method_idx, method in enumerate(METHODS):
        ax = fig.add_subplot(plot_grid[method_idx])
        info = data_group[method]

        if info['found']:
            im = ax.imshow(info['error'], cmap=cmap, vmin=0, vmax=max_error_in_group,
                           aspect='auto', origin='lower', interpolation='nearest',
                           extent=[0, info['t_len'], 0, N_SIZE])
            last_im = im

            label_text = method
            ax.text(0.005, 0.85, label_text, transform=ax.transAxes,
                    fontsize=22, fontweight='bold', color='black',
                    bbox=dict(facecolor='none', alpha=0.9, edgecolor='none', pad=1))
        else:
            ax.text(0.5, 0.5, 'Data Missing', ha='center', color='red')
            ax.set_facecolor('#eeeeee')
            ax.set_xlim(0, info['t_len'])
            ax.set_ylim(0, N_SIZE)

        # === 保持：横纵坐标不显示 0 ===
        fmt = ticker.FuncFormatter(lambda x, pos: f'{int(x)}' if x != 0 else '')
        ax.xaxis.set_major_formatter(fmt)
        ax.yaxis.set_major_formatter(fmt)

        if method_idx == 0:
            title_str = sys_name
            if 'llz' in title_str:
                title_str = title_str.replace('llz', 'Lorenz ')
            elif 'lsl' in title_str:
                title_str = title_str.replace('lsl', 'Rossler ')

            full_title = f"{group_labels[sys_idx]} {title_str}"

            ax.set_title(full_title, fontsize=22, fontweight='bold', pad=3)
            ax.tick_params(labelbottom=False)
        else:
            ax.tick_params(labelbottom=True)

        ax.tick_params(axis='both', which='major', labelsize=16)

    # --- Colorbar ---
    if last_im is not None:
        cbar = fig.colorbar(last_im, cax=cbar_ax)
        cbar.ax.tick_params(labelsize=16)


fig.text(0.04, 0.5, 'Node Index', va='center', rotation='vertical', fontsize=22, fontweight='bold')
fig.text(0.5, 0.02, 'Time Steps', ha='center', fontsize=22, fontweight='bold')
plt.show()

## Figure 7. Evaluation metrics for three coupled dynamical systems


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# === 1. 设置绘图风格 ===
plt.style.use('ggplot')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['grid.color'] = '#EAEAEA'  # 更淡的网格线
plt.rcParams['grid.linestyle'] = ':'    # 点状网格线，模仿参考图
plt.rcParams['axes.unicode_minus'] = False
# 尝试使用衬线字体以贴近论文风格，如果系统没有 Times New Roman 会回退到 Sans
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'SimHei']
plt.rcParams['font.sans-serif'] = ['Arial', 'Microsoft YaHei', 'SimHei']

# === 2. 数据录入 (保持不变) ===
raw_data_input = {
    # 振子数 10
    10: {
        'llz. (x-x)_Error': [0.011811,0.0126729,0.0157313,0.0180233,0.0174851,0.0145468,0.015642,0.0174917,0.0147783,0.0176733,0.01496,0.0142336,0.0143276,0.0156071,	0.0148976,0.0139721,0.0149691,0.0122595,0.0124121,0.016991
],
        'llz. xy_Error':    [0.018992, 0.017044, 0.019502, 0.020378, 0.016298, 0.015926, 0.013527, 0.018652, 0.015507, 0.014562, 0.01798, 0.017167, 0.015587, 0.016305, 0.017338, 0.014037, 0.022162, 0.016611, 0.015218, 0.018063],
        'lsl_Error':        [0.011448, 0.009472, 0.011285, 0.01136, 0.008683, 0.010974, 0.012746, 0.012468, 0.011612, 0.012241, 0.010373, 0.011861, 0.009245, 0.014918, 0.011963, 0.012002, 0.011779, 0.012155, 0.011098, 0.012069],
        'llz. (x-x)_TNR':   [1]*20,
        'llz. xy_TNR':      [1]*20,
        'lsl_TNR':          [1]*20,
        'llz. (x-x)_TPR':   [1]*20,
        'llz. xy_TPR':      [1]*20,
        'lsl_TPR':          [1]*20,
    },
    # 振子数 20
    20: {
        'llz. (x-x)_Error': [0.0152692,0.0170944,0.0183332,0.0183332,0.0139933,0.0153799,	0.0147844,0.0156799,0.0155875,0.0172525,0.0157056,0.0129687,0.0175363,	0.0163902,	0.0132738,0.0159402,0.014933,0.0139189,0.0165434,0.0182515
],
        'llz. xy_Error':    [0.016461, 0.018507, 0.015021, 0.013986, 0.014215, 0.01749, 0.017676, 0.020225, 0.022951, 0.017251, 0.017831, 0.014852, 0.022918, 0.021352, 0.019701, 0.015509, 0.017547, 0.020145, 0.013914, 0.019084],
        'lsl_Error':        [0.01575, 0.017641, 0.01653, 0.017268, 0.016717, 0.018104, 0.016667, 0.016712, 0.016309, 0.017065, 0.019342, 0.018023, 0.017528, 0.017865, 0.017585, 0.015881, 0.017165, 0.018654, 0.015421, 0.016933],
        'llz. (x-x)_TNR':   [1]*20,
        'llz. xy_TNR':      [1]*20,
        'lsl_TNR':          [1]*20,
        'llz. (x-x)_TPR':   [1]*20,
        'llz. xy_TPR':      [1]*20,
        'lsl_TPR':          [1]*20,
    },
    # 振子数 50
    50: {
        'llz. (x-x)_Error': [0.0213778,0.0196725,0.019803,0.0207543,0.019994,0.0209546,	0.021125,0.0197644,0.0210297,0.0213053,0.0205184,0.0205638,0.0201886,0.0203904,0.0199004,0.0185641,0.0203776,0.0198617,0.018515,0.0206284
],
        'llz. xy_Error':    [0.019611, 0.013351, 0.015857, 0.017607, 0.014956, 0.016087, 0.015668, 0.017085, 0.01553, 0.017211, 0.015723, 0.016245, 0.017436, 0.017788, 0.016764, 0.016487, 0.017998, 0.015496, 0.016502, 0.014264],
        'lsl_Error':        [0.016802, 0.016153, 0.016985, 0.016504, 0.01539, 0.017464, 0.015686, 0.017083, 0.016044, 0.017224, 0.016889, 0.015478, 0.015374, 0.015905, 0.015969, 0.016342, 0.016035, 0.016735, 0.015266, 0.016875],
        'llz. (x-x)_TNR':   [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        'llz. xy_TNR':[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0.9982, 1],
        'lsl_TNR': [0.9908	,0.9924,	0.9891	,0.9903	,0.9924	,0.9908	,0.9903	,0.9908	,0.9933,	0.9912,	0.9899,	0.9916,	0.9941	,0.9929,	0.9924,	0.992,	0.9916	,0.992,	0.995,	0.9903],

        'llz. (x-x)_TPR':   [0.9926, 0.9926, 0.9926, 0.9926, 0.9926, 0.9926, 0.9926, 0.9926, 1, 0.9926, 0.9926, 0.9926, 0.9926, 1, 0.9926, 0.9926, 0.9926, 0.9926, 0.9926, 0.9926],
        'llz. xy_TPR':      [1]*20,
        'lsl_TPR':          [1]*20,
    },
    # 振子数 100
  100: {
    'llz. (x-x)_Error': [0.023604,0.0232555,0.02283,0.0235328,0.0237447,0.0223564,0.0235369,0.0228873,0.023258,0.0237557,0.0234888,0.0222056,0.0212056,0.0214056,0.0232545,0.0192856,0.0209316,0.0234647,0.0232358,0.0228347
],
    'llz. xy_Error':    [0.014466, 0.014508, 0.013758, 0.015217, 0.014376, 0.014202, 0.013906, 0.01348, 0.014831, 0.014185, 0.013222, 0.015515, 0.013665, 0.01299, 0.013892, 0.01356, 0.015732, 0.014207, 0.014777, 0.014207],
    'lsl_Error':        [0.0352583, 0.034276, 0.035588, 0.037337, 0.035205, 0.037044, 0.035162, 0.035446, 0.034969, 0.034739, 0.034041, 0.034734, 0.034478, 0.034478, 0.034537, 0.034682, 0.034201, 0.035319, 0.034822, 0.034822],
    'llz. (x-x)_TNR':   [0.9925, 0.9923, 0.993, 0.994, 0.9938, 0.9934, 0.9937, 0.9928, 0.9939, 0.9932, 0.9931, 0.9925, 0.9933, 0.9931, 0.9929, 0.9943, 0.9933, 0.9934, 0.9931, 0.9937],
    'llz. xy_TNR':      [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'lsl_TNR':          [0.9980487,0.9977406,0.9976379,0.9976579,0.999897,0.9987136,0.999897,0.998151,0.997843,0.998357,0.997843,0.999897,0.9976579,0.999897,0.9977406,0.997843,0.9976579,0.999897,0.9980487,0.9967136,0.9976579,0.999897,	0.9987136,0.997843
],
    'llz. (x-x)_TPR':   [0.9935, 0.9935, 0.9935, 0.9935, 0.9935, 0.9935, 0.9968, 0.9935, 0.9935, 0.9935, 0.9935, 0.9935, 0.9935, 0.9931, 0.9935, 0.9935, 0.9935, 0.9968, 0.9935, 0.9935],
    'llz. xy_TPR':      [1, 1, 0.9967, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0.9967, 1, 0.9967, 1, 0.9967, 1, 1, 1],
    'lsl_TPR':          [0.9847909,0.980989,0.980989,0.980989,0.980989,0.980989,0.984791	,0.9847914,0.9847909,0.984791,0.984791,0.9847909,0.984791,0.984791,0.984791,0.9967136,0.9847909,0.9847959,0.9846909,0.9847909
]
    }
}

def plot_from_raw_data(raw_data):
    # 1. 数据预处理
    processed_data = {}
    all_system_names = set()

    for osc, systems in raw_data.items():
        processed_data[osc] = {}
        for key, values in systems.items():
            val_array = np.array(values)
            processed_data[osc][key] = {
                'mean': np.mean(val_array),
                'std': np.std(val_array)
            }
            sys_name = key.split('_')[0]
            all_system_names.add(sys_name)

    x_values = sorted(processed_data.keys())
    print(f"绘图X轴(振子数): {x_values}")

    # 配置：Title对应参考图的标题
    subplots_config = [
        {'title': 'Error',   'keywords': ['error'], 'ylabel': 'Metric Value'},
        {'title': 'TPR', 'keywords': ['tpr'],   'ylabel': ''},
        {'title': 'TNR', 'keywords': ['tnr'],   'ylabel': ''}
    ]
    # 对应的 a, b, c 标签
    abc_labels = ['(a)', '(b)', '(c)']

    # 系统名排序: llz. (x-x), llz. xy, lsl
    sorted_sys_names = sorted(list(all_system_names))

    # 颜色配置 (高饱和度)
    colors = ['#008db8', '#e8ac41', '#b9514e'] # 调整为更接近参考图的 深蓝/黄/红 或者是之前要求的 青/橙/紫?
    # 这里为了匹配"设计感"，使用稍微调整过的对比色，或者沿用您上一轮满意的：
    colors = ['#00BFC4', '#FF8C00', '#8A2BE2'] # 保持您要求的"饱和度高"

    # 标记配置 (参考图：圆圈, 菱形, 正方形)
    markers = ['o', 'D', 's']

    style_map = {name: {'marker': markers[i % len(markers)], 'color': colors[i % len(colors)]}
                 for i, name in enumerate(sorted_sys_names)}

    # 名称映射 (Labels) - 使用 Lorenz. (x-x) 格式
    label_map = {
        'llz. (x-x)': 'Lorenz.(x-x)',
        'llz. xy': 'Lorenz.(xy)',
        'lsl': 'Rossler.(x-x)'
    }

    # === 【关键修改：尺寸调整为宽条形，类似参考图】 ===
    fig, axes = plt.subplots(1, 3, figsize=(20, 5), constrained_layout=True)

    for i, config in enumerate(subplots_config):
        ax = axes[i]
        plotted_labels = []

        for sys_name in sorted_sys_names:
            means = []
            stds = []
            valid_x = []

            for x in x_values:
                candidates = processed_data[x].keys()
                target_key = None
                for k in candidates:
                    if k.startswith(sys_name + '_') and any(kw in k.lower() for kw in config['keywords']):
                        target_key = k
                        break

                if target_key:
                    means.append(processed_data[x][target_key]['mean'])
                    stds.append(processed_data[x][target_key]['std'])
                    valid_x.append(x)

            if valid_x:
                style = style_map[sys_name]
                display_label = label_map.get(sys_name, sys_name)

                # === 【关键修改：空心标记点效果】 ===
                ax.errorbar(valid_x, means, yerr=stds,
                            fmt=f"-{style['marker']}",
                            color=style['color'],
                            label=display_label,
                            capsize=4,
                            linewidth=2.5,        # 线条稍细一点，显得更精致
                            markersize=9,         # 标记点适中
                            markeredgewidth=2,    # 标记边缘宽度
                            markerfacecolor='white', # 关键：白色填充，实现空心效果
                            alpha=0.9,
                            elinewidth=2)
                plotted_labels.append(display_label)

        # === 设置边框 (Spines) 为深灰色 ===
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color('#333333')
            spine.set_linewidth(1.5)

        # === 标题和标签 ===
        ax.set_title(config['title'], fontsize=18, fontweight='bold', pad=12)
        ax.set_xlabel('Network Size ($M$)', fontsize=18, fontweight='bold') # 使用 LaTeX N

        # 仅在第一个图显示 Y轴 Label
        if config['ylabel']:
            ax.set_ylabel(config['ylabel'], fontsize=18, fontweight='bold')

        # === 【关键修改：添加 (a) (b) (c)】 ===
        # 坐标 (-0.15, 1.08) 是相对于子图左上角的偏移量，根据需要微调
        ax.text(-0.12, 1.08, abc_labels[i], transform=ax.transAxes,
                fontsize=18, fontweight='bold', va='bottom', ha='right')

        # 纵轴范围
        if 'Error' in config['title']:
            ax.set_ylim(bottom=0,top=0.04)
        else:
            # TPR/TNR 参考图是从较高位置开始
            ax.set_ylim(bottom=0.92, top=1.01)

        ax.set_xticks(x_values)
        ax.tick_params(axis='both', which='major', labelsize=25, width=1.5, colors='#333333')
        ax.grid(True, linestyle=':', alpha=0.6) # 点状网格

        # 图例：仅在第一个图显示，且放在左上角内部（参考图样式）
        if i == 0 and plotted_labels:
            ax.legend(loc='upper left', fontsize=16, frameon=True, edgecolor='#333333')

    output_filename = 'error、TPR、TNR.pdf'
    plt.savefig(output_filename, dpi=600, bbox_inches='tight') # bbox_inches='tight' 防止标签被切
    print(f"✅ 图表已保存为: {output_filename}")
    plt.show()

if __name__ == "__main__":
    plot_from_raw_data(raw_data_input)

## Figure 8. Valid prediction steps for the two methods


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# === 1. 设置绘图风格 (Times New Roman + 黑色字体) ===
plt.style.use('default')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'

# === 2. 数据录入 ===
prc_lsl = [5891, 3469, 4686, 6471, 5297, 4709, 4690, 4702, 2322, 4697, 5894, 5899, 3485, 6449, 1718, 3519, 3474, 4684, 3472, 4687]
prc_llz_xx = [306, 460, 462, 297, 481, 451, 752, 300, 299, 295, 298, 565, 299, 571, 302, 295, 293, 302, 302, 295]
prc_llz_xy = [117, 70, 111, 79, 104, 97, 81, 88, 109, 125, 125, 104, 134, 148, 91, 103, 73, 119, 103, 88]

pci_llz_xx = [302, 365, 482, 486, 304, 301, 452, 481, 481, 482, 486, 371, 487, 370, 471, 485, 460, 299, 293, 365]
pci_llz_xy = [329, 382, 337, 380, 340, 331, 427, 380, 330, 326, 331, 336, 341, 333, 348, 344, 347, 326, 332, 329]
pci_lsl = [4694, 5899, 2885, 4669, 4104, 3485, 5886, 6471, 5298, 4668, 4685, 6460, 3481, 6456, 5293, 6455, 2873, 3465, 5253, 6452]

data_groups = [
    [prc_llz_xx, pci_llz_xx],
    [prc_llz_xy, pci_llz_xy],
    [prc_lsl, pci_lsl]
]
systems = ['Lorenz(x-x)', 'Lorenz(xy)', 'Rossler(x-x)']
colors = ['#D4A135', '#004D73'] # 红绿配色
abc_labels = ['(a)', '(b)', '(c)'] # 子图标签

# === 3. 绘图配置 ===
fig, axes = plt.subplots(1, 3, figsize=(12, 6), constrained_layout=True)

width = 0.5
positions = [1, 2]

for i, ax in enumerate(axes):
    sys_name = systems[i]
    group_data = data_groups[i]

    # 绘制 PRC 箱线图
    ax.boxplot(group_data[0], positions=[positions[0]], widths=width,
               patch_artist=True,
               boxprops=dict(facecolor=colors[0], color=colors[0], linewidth=1),
               capprops=dict(color='black', linewidth=1),
               whiskerprops=dict(color='black', linewidth=1),
               flierprops=dict(markeredgecolor='black', marker='o', markersize=4, linewidth=1),
               medianprops=dict(color='#FF8C00', linewidth=1.5))

    # 绘制 PCI-RC 箱线图
    ax.boxplot(group_data[1], positions=[positions[1]], widths=width,
               patch_artist=True,
               boxprops=dict(facecolor=colors[1], color=colors[1], linewidth=1),
               capprops=dict(color='black', linewidth=1),
               whiskerprops=dict(color='black', linewidth=1),
               flierprops=dict(markeredgecolor='black', marker='s', markersize=4, linewidth=1),
               medianprops=dict(color='#FF8C00', linewidth=1.5))

    # === 坐标轴与装饰 ===
    ax.set_title(sys_name, fontsize=20, fontweight='bold', pad=10, family='Times New Roman')
    ax.set_xticks(positions)
    ax.set_xticklabels(['PRC', 'PCI-RC'], fontsize=20, fontweight='normal', family='Times New Roman')
    ax.set_xlim(0.5, 2.5)

    # Y轴设置 (科学计数法)
    ax.tick_params(axis='y', labelsize=25)
    ax.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

    # 边框
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#444444')
        spine.set_linewidth(1.5)

    # 网格
    ax.grid(True, which="major", axis='y', linestyle='--', alpha=0.5, color='gray')


    ax.text(-0.1, 1.05, abc_labels[i], transform=ax.transAxes,
            fontsize=18, fontweight='bold', va='bottom', ha='right',
            family='Times New Roman')

# 仅给第一个图加 Y 轴标签
axes[0].set_ylabel('VPS', fontsize=20, fontweight='normal', family='Times New Roman')

output_filename = 'prediction_vps.pdf'
plt.savefig(output_filename, dpi=600)
print(f"✅ 图表已保存为: {output_filename}")
plt.show()

## Utilities for Figures 2, 6, 11–16, and 18–23


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Shared configuration for adjacency-matrix error visualizations
# ============================================================
DIR_TRUE = 'generate.data'
DIR_PRED = 'result.date'
V_MIN, V_MAX = 0.0, 0.10
ERROR_CMAP = 'cividis'

SYSTEM_LABELS = {
    'llz(x-x)': r'Lorenz $(x-x)$',
    'llz(x.y)': r'Lorenz $(x.y)$',
    'lsl(x-x)': r'Rossler $(x-x)$'
}

TRUE_PATH_TEMPLATES = {
    'llz(x-x)': 'llz.X-X/{N}coupled_lorenz_adj_thresholded.npy',
    'llz(x.y)': 'llz.xy/{N}coupled_lorenz_adj_thresholded.xy.npy',
    'lsl(x-x)': 'lsl/{N}coupled_rossler_adj_thresholded.npy'
}

PRED_PATH_TEMPLATES = {
    'unknown': {
        'llz(x-x)': 'llz.(x-x)/network/{N}.(X-X)unknow_W_predicted.npy',
        'llz(x.y)': 'llz.xy/network/{N}.(XY)unknow_W_predicted.npy',
        'lsl(x-x)': 'lsl/network/{N}.lsl.unknow_y_adj_traj.npy'
    },
    'known': {
        'llz(x-x)': 'llz.(x-x)/network/{N}know_W_predicted.(x-x).npy',
        'llz(x.y)': 'llz.xy/network/{N}know_W_predicted.(xy).npy',
        'lsl(x-x)': 'lsl/network/{N}rossler_know_W_predicted.npy'
    }
}


def get_error_matrix_paths(system_name, N, mode):
    """Return the true and predicted adjacency-matrix paths."""
    true_path = os.path.join(
        DIR_TRUE,
        TRUE_PATH_TEMPLATES[system_name].format(N=N)
    )
    pred_path = os.path.join(
        DIR_PRED,
        PRED_PATH_TEMPLATES[mode][system_name].format(N=N)
    )
    return true_path, pred_path


def load_absolute_error_matrix(system_name, N, mode, *, use_all_dimensions=True):
    """Load two adjacency matrices and return |W_true - W_pred|."""
    true_path, pred_path = get_error_matrix_paths(system_name, N, mode)
    error_matrix = np.zeros((N, N))

    try:
        if os.path.exists(true_path) and os.path.exists(pred_path):
            w_true = np.load(true_path)
            w_pred = np.load(pred_path)

            if use_all_dimensions:
                # Original M=10/20 grid code used all four matrix dimensions.
                n = min(
                    w_true.shape[0],
                    w_true.shape[1],
                    w_pred.shape[0],
                    w_pred.shape[1]
                )
            else:
                # Original M=50/100 single-panel code used the first dimension.
                n = min(
                    w_true.shape[0],
                    w_pred.shape[0]
                )

            error_matrix = np.abs(
                w_true[:n, :n] - w_pred[:n, :n]
            )
        else:
            print(f'File missing: {system_name}, M={N}')
            print(f'True file: {true_path}')
            print(f'Predicted file: {pred_path}')

    except Exception as error:
        print(f'Error loading {system_name}, M={N}: {error}')

    return error_matrix


# ============================================================
# 3 x 2 matrix figures for M = 10 and M = 20
# All style values are preserved from the original notebook.
# ============================================================
GRID_STYLE = {
    'unknown': {
        'rc_font_size': 15,
        'axes_linewidth': 1.2,
        'figsize': (12, 15),
        'adjust': dict(
            left=0.075, right=0.875, top=0.975, bottom=0.015,
            wspace=0.015, hspace=0.015
        ),
        'spine_width': 1.2,
        'spine_color': '#444444',
        'title_fontsize': 24,
        'title_pad': 6,
        'ylabel_fontsize': 22,
        'ylabel_pad': 7,
        'cbar_label_fontsize': 19,
        'cbar_labelpad': 15,
        'cbar_tick_params': dict(labelsize=17, width=1.2, length=5),
        'set_cbar_ticks': True,
        'output': 'unknow.error_matrices.pdf'
    },
    'known': {
        'rc_font_size': 12,
        'axes_linewidth': 1.0,
        'figsize': (11, 13.5),
        'adjust': dict(
            left=0.08, right=0.88, top=0.97, bottom=0.02,
            wspace=0.02, hspace=0.02
        ),
        'spine_width': 1.0,
        'spine_color': '#555555',
        'title_fontsize': 20,
        'title_pad': 5,
        'ylabel_fontsize': 19,
        'ylabel_pad': 6,
        'cbar_label_fontsize': 17,
        'cbar_labelpad': 14,
        'cbar_tick_params': dict(labelsize=15),
        'set_cbar_ticks': False,
        'output': 'know_error_matrices.pdf'
    }
}


def plot_error_matrix_grid(mode):
    """Reproduce the original 3 x 2 error-matrix figure for M=10 and M=20."""
    style = GRID_STYLE[mode]
    sizes = [10, 20]
    systems = ['llz(x-x)', 'llz(x.y)', 'lsl(x-x)']

    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman'],
        'mathtext.fontset': 'stix',
        'font.size': style['rc_font_size'],
        'axes.linewidth': style['axes_linewidth'],
        'figure.dpi': 600
    })

    # The original unknown-network cell closed previous figures first.
    if mode == 'unknown':
        plt.close('all')

    fig, axes = plt.subplots(3, 2, figsize=style['figsize'])
    plt.subplots_adjust(**style['adjust'])

    cmap = plt.get_cmap(ERROR_CMAP)
    last_im = None

    for i, system_name in enumerate(systems):
        for j, N in enumerate(sizes):
            ax = axes[i, j]
            error_matrix = load_absolute_error_matrix(
                system_name,
                N,
                mode,
                use_all_dimensions=True
            )

            last_im = ax.imshow(
                error_matrix,
                cmap=cmap,
                vmin=V_MIN,
                vmax=V_MAX,
                aspect='equal',
                interpolation='nearest'
            )

            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_anchor('C')

            for spine in ax.spines.values():
                spine.set_linewidth(style['spine_width'])
                spine.set_edgecolor(style['spine_color'])

            if i == 0:
                ax.set_title(
                    rf'$M={N}$',
                    fontsize=style['title_fontsize'],
                    fontweight='bold',
                    pad=style['title_pad']
                )

            if j == 0:
                ax.set_ylabel(
                    SYSTEM_LABELS[system_name],
                    fontsize=style['ylabel_fontsize'],
                    fontweight='bold',
                    labelpad=style['ylabel_pad']
                )

    cbar_ax = fig.add_axes([0.91, 0.20, 0.018, 0.60])
    cbar = fig.colorbar(last_im, cax=cbar_ax)

    cbar.set_label(
        r'Absolute Error $|W_{\mathrm{true}}-W_{\mathrm{pred}}|$',
        fontsize=style['cbar_label_fontsize'],
        fontweight='bold',
        labelpad=style['cbar_labelpad']
    )
    cbar.ax.tick_params(**style['cbar_tick_params'])

    if style['set_cbar_ticks']:
        cbar.set_ticks(np.linspace(V_MIN, V_MAX, 6))

    fig.savefig(
        style['output'],
        format='pdf',
        dpi=600,
        bbox_inches='tight'
    )
    plt.show()


# ============================================================
# Individual matrix figures for M = 50 and M = 100
# ============================================================
def get_single_error_output(system_name, N, mode):
    """Preserve the original PDF filenames exactly."""
    prefix = 'k' if mode == 'known' else 'un'

    if system_name == 'llz(x-x)':
        middle = 'Lorenz_x-x'
    elif system_name == 'llz(x.y)':
        # The original notebook used x-y for M=50 and x.y for M=100.
        middle = 'Lorenz_x-y' if N == 50 else 'Lorenz_x.y'
    else:
        middle = 'Rossler_x-x'

    return f'{prefix}_absolute_error_{middle}_M{N}.pdf'


def plot_single_error_matrices(N, mode):
    """
    Reproduce the original individual error-matrix PDFs.

    The M=100 unknown-network figure intentionally keeps its larger layout,
    title sizes, and colorbar dimensions from the original code.
    """
    systems = ['llz(x-x)', 'llz(x.y)', 'lsl(x-x)']
    cmap = plt.get_cmap(ERROR_CMAP)

    is_large_unknown = (N == 100 and mode == 'unknown')

    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman'],
        'mathtext.fontset': 'stix',
        'font.size': 12 if is_large_unknown else 11,
        'axes.linewidth': 1.0,
        'figure.dpi': 600
    })

    for system_name in systems:
        error_matrix = load_absolute_error_matrix(
            system_name,
            N,
            mode,
            use_all_dimensions=False
        )

        if is_large_unknown:
            # Exact layout from the original M=100 unknown-network cell.
            fig, ax = plt.subplots(figsize=(8.5, 7.5))
            plt.subplots_adjust(
                left=0.06,
                right=0.82,
                top=0.88,
                bottom=0.06
            )

            fig.suptitle(
                rf'$M={N}$',
                fontsize=22,
                fontweight='bold',
                y=0.97
            )

            im = ax.imshow(
                error_matrix,
                cmap=cmap,
                vmin=V_MIN,
                vmax=V_MAX,
                aspect='equal',
                interpolation='nearest'
            )

            ax.set_title(
                SYSTEM_LABELS[system_name],
                fontsize=18,
                fontweight='bold',
                pad=5
            )
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_anchor('C')

            for spine in ax.spines.values():
                spine.set_linewidth(1.0)
                spine.set_edgecolor('#555555')

            cbar_ax = fig.add_axes([0.86, 0.15, 0.03, 0.70])
            cbar = fig.colorbar(im, cax=cbar_ax)

            cbar.set_label(
                r'Absolute Error $|W_{\mathrm{true}}-W_{\mathrm{pred}}|$',
                fontsize=16,
                fontweight='bold',
                labelpad=14
            )
            cbar.ax.tick_params(labelsize=14)
            cbar.set_ticks(np.linspace(V_MIN, V_MAX, 6))

            save_kwargs = {}

        else:
            # Exact layout used by M=50 unknown/known and M=100 known.
            fig = plt.figure(figsize=(7.2, 6.5))
            ax = fig.add_axes([0.06, 0.08, 0.75, 0.75])

            im = ax.imshow(
                error_matrix,
                cmap=cmap,
                vmin=V_MIN,
                vmax=V_MAX,
                aspect='equal',
                interpolation='nearest'
            )

            ax.set_xticks([])
            ax.set_yticks([])

            for spine in ax.spines.values():
                spine.set_linewidth(1.0)
                spine.set_edgecolor('#555555')

            ax.set_title(
                SYSTEM_LABELS[system_name],
                fontsize=16,
                fontweight='bold',
                pad=8
            )

            fig.text(
                0.435,
                0.93,
                rf'$M={N}$',
                ha='center',
                va='center',
                fontsize=18,
                fontweight='bold'
            )

            cbar_ax = fig.add_axes([0.84, 0.14, 0.025, 0.63])
            cbar = fig.colorbar(im, cax=cbar_ax)
            cbar.set_ticks(np.linspace(V_MIN, V_MAX, 6))

            cbar.ax.tick_params(
                labelsize=11,
                width=0.8,
                length=3
            )
            cbar.set_label(
                r'Absolute Error $|W_{\mathrm{true}}-W_{\mathrm{pred}}|$',
                fontsize=12,
                fontweight='bold',
                labelpad=10
            )

            save_kwargs = {'pad_inches': 0.05}

        output_file = get_single_error_output(system_name, N, mode)

        fig.savefig(
            output_file,
            format='pdf',
            dpi=600,
            bbox_inches='tight',
            **save_kwargs
        )

        print(f'Saved: {output_file}')
        plt.show()
        plt.close(fig)


## Figures 2, 6, 11–16, and 18–23. Weight reconstruction error matrices

- **Figure 2:** Known network topology, $M=10$ and $20$
- **Figure 6:** Unknown network topology, $M=10$ and $20$
- **Figures 11–16:** Known network topology, $M=50$ and $100$
- **Figures 18–23:** Unknown network topology, $M=50$ and $100$


In [ ]:
# 1. M = 10 and M = 20, unknown network information
plot_error_matrix_grid('unknown')

# 2. M = 10 and M = 20, known network information
plot_error_matrix_grid('known')

# 3. M = 50, three separate unknown-network PDFs
plot_single_error_matrices(50, 'unknown')

# 4. M = 50, three separate known-network PDFs
plot_single_error_matrices(50, 'known')

# 5. M = 100, three separate unknown-network PDFs
plot_single_error_matrices(100, 'unknown')

# 6. M = 100, three separate known-network PDFs
plot_single_error_matrices(100, 'known')


## Utilities for Figures 5 and 17

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
from matplotlib.lines import Line2D


def classify_reconstruction_graph(w_true, w_pred, pred_threshold=0.0):
    """
    Build the directed reconstruction graph and classify edges as
    TP (correct), FP (spurious), or FN (missed).
    """
    n = min(
        w_true.shape[0],
        w_true.shape[1],
        w_pred.shape[0],
        w_pred.shape[1]
    )
    w_true = w_true[:n, :n]
    w_pred = w_pred[:n, :n]

    graph = nx.DiGraph()
    graph.add_nodes_from(range(n))

    edges = {
        'TP': [],
        'FP': [],
        'FN': []
    }
    tp_weights = []

    rows, cols = np.where(
        (w_true > 0) | (w_pred > pred_threshold)
    )

    for row, col in zip(rows, cols):
        if row == col:
            continue

        true_edge = w_true[row, col] > 0
        pred_edge = w_pred[row, col] > pred_threshold

        if true_edge and pred_edge:
            edge_type = 'TP'
            weight = float(w_pred[row, col])
            tp_weights.append(weight)
        elif pred_edge:
            edge_type = 'FP'
            weight = float(w_pred[row, col])
        elif true_edge:
            edge_type = 'FN'
            weight = float(w_true[row, col])
        else:
            continue

        edges[edge_type].append((row, col))
        graph.add_edge(
            row,
            col,
            type=edge_type,
            weight=weight
        )

    return graph, edges, tp_weights


def draw_classified_edges(
    ax,
    graph,
    pos,
    edges,
    tp_weights,
    *,
    cmap_tp,
    tp_min,
    tp_max,
    color_fp,
    color_fn,
    node_size,
    tp_style,
    fp_style,
    fn_style
):
    """Draw TP, FP, and FN edges using caller-provided visual styles."""
    if edges['TP']:
        nx.draw_networkx_edges(
            graph,
            pos,
            ax=ax,
            edgelist=edges['TP'],
            edge_color=tp_weights,
            edge_cmap=cmap_tp,
            edge_vmin=tp_min,
            edge_vmax=tp_max,
            node_size=node_size,
            arrows=True,
            **tp_style
        )

    if edges['FP']:
        nx.draw_networkx_edges(
            graph,
            pos,
            ax=ax,
            edgelist=edges['FP'],
            edge_color=color_fp,
            node_size=node_size,
            arrows=True,
            **fp_style
        )

    if edges['FN']:
        nx.draw_networkx_edges(
            graph,
            pos,
            ax=ax,
            edgelist=edges['FN'],
            edge_color=color_fn,
            node_size=node_size,
            arrows=True,
            **fn_style
        )


## Figure 5. Reconstructed network topologies


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'stix',
    'font.size': 15,
    'figure.dpi': 110,
    'savefig.dpi': 300
})

TRUE_DIR = Path('generate.data')
PRED_DIR = Path('result.date')

SIZES = [10, 20, 50, 100]

SYSTEMS = [
    (
        'llz(x-x)',
        r'Lorenz $(x-x)$',
        'llz.X-X/{N}coupled_lorenz_adj_thresholded.npy',
        'llz.(x-x)/network/{N}.(X-X)unknow_W_predicted.npy'
    ),
    (
        'llz(x.y)',
        r'Lorenz $(x.y)$',
        'llz.xy/{N}coupled_lorenz_adj_thresholded.xy.npy',
        'llz.xy/network/{N}.(XY)unknow_W_predicted.npy'
    ),
    (
        'lsl(x-x)',
        r'Rossler $(x-x)$',
        'lsl/{N}coupled_rossler_adj_thresholded.npy',
        'lsl/network/{N}.lsl.unknow_y_adj_traj.npy'
    )
]

GROUP_LABELS = ['(a)', '(b)', '(c)']

CMAP_TP = plt.cm.Blues
COLOR_FP = '#FFC20A'
COLOR_FN = '#D62728'
NODE_COLOR = '#4A4845'

TP_MIN = 0.0
TP_MAX = 0.9
PRED_THRESHOLD = 0.0

EDGE_WIDTH = 1.6
ARROW_SIZE = 10
EDGE_ALPHA = 0.85

NODE_SIZE = {
    10: 110,
    20: 80,
    50: 45,
    100: 24
}

CURVATURE = {
    10: 0.07,
    20: 0.06,
    50: 0.035,
    100: 0.015
}

PADDING = {
    10: 0.10,
    20: 0.08,
    50: 0.06,
    100: 0.06
}


# ============================================================
# 2. Load network data
# ============================================================
def load_graph_for_size(key, N, true_file, pred_file):
    true_path = TRUE_DIR / true_file.format(N=N)
    pred_path = PRED_DIR / pred_file.format(N=N)

    if not true_path.exists():
        print(f'True file missing: {true_path}')
        return None, None, None

    if not pred_path.exists():
        print(f'Predicted file missing: {pred_path}')
        return None, None, None

    try:
        w_true = np.load(true_path)
        w_pred = np.load(pred_path)
        graph, edges, tp_weights = classify_reconstruction_graph(
            w_true,
            w_pred,
            pred_threshold=PRED_THRESHOLD
        )
    except Exception as error:
        print(f'Error loading {key}, M={N}: {error}')
        return None, None, None

    print(
        f'{key}, M={N}: '
        f'TP={len(edges["TP"])}, '
        f'FP={len(edges["FP"])}, '
        f'FN={len(edges["FN"])}'
    )
    return graph, edges, tp_weights


# ============================================================
# 3. Layout functions
# ============================================================
def uniform_disk_layout(graph, seed):
    nodes = sorted(
        graph.nodes(),
        key=lambda node: graph.degree(node),
        reverse=True
    )
    count = len(nodes)

    if count == 0:
        return {}

    index = np.arange(count)
    radius = np.sqrt((index + 0.5) / count)

    golden_angle = np.pi * (3 - np.sqrt(5))
    phase = np.deg2rad(seed % 360)
    angle = index * golden_angle + phase

    coordinates = np.column_stack((
        radius * np.cos(angle),
        radius * np.sin(angle)
    ))

    position_order = np.argsort(
        np.abs(radius - 0.70)
    )

    return {
        node: coordinates[position]
        for node, position in zip(nodes, position_order)
    }


def spring_layout(graph, N, seed):
    layout_graph = nx.Graph(graph)

    components = sorted(
        nx.connected_components(layout_graph),
        key=len,
        reverse=True
    )

    if len(components) > 1:
        main_nodes = list(components[0])

        for index, component in enumerate(components[1:]):
            layout_graph.add_edge(
                main_nodes[index % len(main_nodes)],
                next(iter(component))
            )

    return nx.spring_layout(
        layout_graph,
        k=2.8 / np.sqrt(N),
        seed=seed,
        iterations=500,
        scale=1.0,
        weight=None
    )


def get_layout(graph, key, N, seed):
    if N == 100 and key == 'llz(x-x)':
        return uniform_disk_layout(graph, seed)

    return spring_layout(graph, N, seed)


# ============================================================
# 4. Draw the 4 x 3 network figure
# ============================================================
plt.close('all')

fig, axes = plt.subplots(
    4,
    3,
    figsize=(16, 21)
)

plt.subplots_adjust(
    left=0.065,
    right=0.90,
    top=0.96,
    bottom=0.09,
    wspace=0.12,
    hspace=0.16
)

for row_index, N in enumerate(SIZES):
    for col_index, system in enumerate(SYSTEMS):
        key, title, true_file, pred_file = system
        ax = axes[row_index, col_index]

        if row_index == 0:
            ax.set_title(
                f'{GROUP_LABELS[col_index]} {title}',
                fontsize=22,
                fontweight='bold',
                pad=10
            )

        if col_index == 0:
            ax.text(
                -0.065,
                0.5,
                rf'$M={N}$',
                transform=ax.transAxes,
                fontsize=22,
                fontweight='bold',
                ha='center',
                va='center',
                rotation=90
            )

        graph, edges, tp_weights = load_graph_for_size(
            key,
            N,
            true_file,
            pred_file
        )

        if graph is None:
            ax.text(
                0.5,
                0.5,
                'Data Missing',
                ha='center',
                va='center',
                fontsize=16
            )
            ax.axis('off')
            continue

        seed = 42 + row_index * 10 + col_index * 100
        pos = get_layout(graph, key, N, seed)

        common_edge_style = {
            'width': EDGE_WIDTH,
            'alpha': EDGE_ALPHA,
            'arrowsize': ARROW_SIZE,
            'connectionstyle': f'arc3,rad={CURVATURE[N]}'
        }

        draw_classified_edges(
            ax,
            graph,
            pos,
            edges,
            tp_weights,
            cmap_tp=CMAP_TP,
            tp_min=TP_MIN,
            tp_max=TP_MAX,
            color_fp=COLOR_FP,
            color_fn=COLOR_FN,
            node_size=NODE_SIZE[N],
            tp_style=common_edge_style,
            fp_style=common_edge_style,
            fn_style=common_edge_style
        )

        nx.draw_networkx_nodes(
            graph,
            pos,
            ax=ax,
            node_size=NODE_SIZE[N],
            node_color=NODE_COLOR,
            edgecolors='#B0B0B0',
            linewidths=0.8
        )

        coordinates = np.array(list(pos.values()))
        x_min, y_min = coordinates.min(axis=0)
        x_max, y_max = coordinates.max(axis=0)

        x_range = x_max - x_min
        y_range = y_max - y_min

        x_pad = max(x_range * PADDING[N], 0.05)
        y_pad = max(y_range * PADDING[N], 0.05)

        ax.set_xlim(x_min - x_pad, x_max + x_pad)
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
        ax.axis('off')


# ============================================================
# 5. Colorbar and legend
# ============================================================
normalizer = mcolors.Normalize(
    vmin=TP_MIN,
    vmax=TP_MAX
)

scalar_map = plt.cm.ScalarMappable(
    cmap=CMAP_TP,
    norm=normalizer
)
scalar_map.set_array([])

colorbar_ax = fig.add_axes([
    0.925,
    0.27,
    0.014,
    0.46
])

colorbar = fig.colorbar(
    scalar_map,
    cax=colorbar_ax
)

colorbar.set_label(
    'TP Weight',
    fontsize=18,
    fontweight='bold',
    labelpad=12
)
colorbar.ax.tick_params(labelsize=16)

legend_elements = [
    Line2D(
        [0], [0],
        color=CMAP_TP(0.75),
        linewidth=EDGE_WIDTH,
        label='Correct (TP)'
    ),
    Line2D(
        [0], [0],
        color=COLOR_FP,
        linewidth=EDGE_WIDTH,
        label='Spurious (FP)'
    ),
    Line2D(
        [0], [0],
        color=COLOR_FN,
        linewidth=EDGE_WIDTH,
        label='Missed (FN)'
    ),
    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        markerfacecolor=NODE_COLOR,
        markeredgecolor='#B0B0B0',
        markersize=12,
        label='Node'
    )
]

fig.legend(
    handles=legend_elements,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.025),
    ncol=4,
    frameon=False,
    fontsize=17,
    columnspacing=2.2,
    handlelength=2.2
)

fig.savefig(
    'network_all_sizes_final.pdf',
    dpi=600,
    facecolor='white'
)

plt.show()


## Figure 17. Reconstructed network topology for the nonlinear product-coupled Lorenz $(x\cdot y)$ system with $M=700$


In [ ]:
# ============================================================
# 1. Large-network configuration
# ============================================================
TRUE_DIR = Path('generate.data')
PRED_DIR = Path('result.date')
N = 700

TRUE_FILE = TRUE_DIR / 'llz.xy' / f'{N}coupled_lorenz_adj_thresholded.xy.npy'
PRED_FILE = PRED_DIR / 'llz.xy' / 'network' / f'{N}.(XY)unknow_W_predicted.npy'
OUTPUT_FILE = '700_lorenz_xy_network.pdf'

CMAP_TP = plt.cm.Blues
COLOR_FP, COLOR_FN, NODE_COLOR = '#FFC20A', '#D62728', '#222222'
TP_MIN, TP_MAX, PRED_THRESHOLD = 0.0, 0.9, 0.0

NODE_EDGE_WIDTH = 0.55
TP_EDGE_WIDTH, TP_EDGE_ALPHA = 0.95, 0.62
ERROR_EDGE_WIDTH, ERROR_EDGE_ALPHA = 1.45, 0.88
ARROW_SIZE_TP, ARROW_SIZE_ERROR = 5.5, 6.8
N_RINGS, RING_GAP = 12, 1.0
FIG_SIZE = (32, 32)


def load_large_graph():
    if not TRUE_FILE.exists() or not PRED_FILE.exists():
        raise FileNotFoundError(
            f'文件不存在：\n{TRUE_FILE}\n{PRED_FILE}'
        )

    w_true = np.load(TRUE_FILE)
    w_pred = np.load(PRED_FILE)

    graph, edges, tp_weights = classify_reconstruction_graph(
        w_true,
        w_pred,
        pred_threshold=PRED_THRESHOLD
    )

    print(
        f'Nodes={graph.number_of_nodes()}, '
        f'TP={len(edges["TP"])}, '
        f'FP={len(edges["FP"])}, '
        f'FN={len(edges["FN"])}'
    )
    return graph, edges, tp_weights


def ring_layout(graph):
    nodes = sorted(graph.nodes())
    count = len(nodes)

    radii = np.arange(1, N_RINGS + 1, dtype=float) * RING_GAP
    raw = count * radii / radii.sum()
    counts = np.floor(raw).astype(int)
    counts[counts < 1] = 1

    diff = count - counts.sum()
    if diff > 0:
        order = np.argsort(raw - np.floor(raw))[::-1]
        counts[order[:diff]] += 1
    elif diff < 0:
        for i in np.argsort(counts)[::-1]:
            while diff < 0 and counts[i] > 1:
                counts[i] -= 1
                diff += 1

    pos, k = {}, 0
    for i, (radius, num) in enumerate(zip(radii, counts)):
        phase = 0 if i % 2 == 0 else np.pi / num
        angles = np.linspace(
            0,
            2 * np.pi,
            num,
            endpoint=False
        ) + phase

        for angle in angles:
            if k >= count:
                break

            pos[nodes[k]] = np.array([
                radius * np.cos(angle),
                radius * np.sin(angle)
            ])
            k += 1

    return pos


def plot_large_lorenz_xy_network(
    *,
    base_fontsize,
    node_size,
    colorbar_fontsize,
    legend_fontsize,
    close_before
):
    """
    Generate one of the two original M=700 visual variants.

    Both calls intentionally save to the same PDF name because this matches
    the original notebook, where the large-font version overwrites the
    standard-font version when all cells are run in order.
    """
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman'],
        'mathtext.fontset': 'stix',
        'font.size': base_fontsize,
        'figure.dpi': 120,
        'savefig.dpi': 600
    })

    if close_before:
        plt.close('all')

    graph, edges, tp_weights = load_large_graph()
    pos = ring_layout(graph)

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    plt.subplots_adjust(
        left=0.025,
        right=0.885,
        top=0.975,
        bottom=0.085
    )

    draw_classified_edges(
        ax,
        graph,
        pos,
        edges,
        tp_weights,
        cmap_tp=CMAP_TP,
        tp_min=TP_MIN,
        tp_max=TP_MAX,
        color_fp=COLOR_FP,
        color_fn=COLOR_FN,
        node_size=node_size,
        tp_style={
            'width': TP_EDGE_WIDTH,
            'alpha': TP_EDGE_ALPHA,
            'arrowsize': ARROW_SIZE_TP,
            'connectionstyle': 'arc3,rad=0.008'
        },
        fp_style={
            'width': ERROR_EDGE_WIDTH,
            'alpha': ERROR_EDGE_ALPHA,
            'arrowsize': ARROW_SIZE_ERROR,
            'connectionstyle': 'arc3,rad=0.025'
        },
        fn_style={
            'width': ERROR_EDGE_WIDTH,
            'alpha': ERROR_EDGE_ALPHA,
            'arrowsize': ARROW_SIZE_ERROR,
            'connectionstyle': 'arc3,rad=-0.025'
        }
    )

    nx.draw_networkx_nodes(
        graph,
        pos,
        ax=ax,
        node_size=node_size,
        node_color=NODE_COLOR,
        edgecolors='white',
        linewidths=NODE_EDGE_WIDTH
    )

    outer = N_RINGS * RING_GAP
    ax.set(
        xlim=(-outer - 0.8, outer + 0.8),
        ylim=(-outer - 0.8, outer + 0.8)
    )
    ax.set_aspect('equal')
    ax.axis('off')

    scalar_map = plt.cm.ScalarMappable(
        cmap=CMAP_TP,
        norm=mcolors.Normalize(TP_MIN, TP_MAX)
    )
    scalar_map.set_array([])

    cax = fig.add_axes([0.91, 0.27, 0.014, 0.46])
    cbar = fig.colorbar(scalar_map, cax=cax)
    cbar.set_label(
        'TP Weight',
        fontsize=colorbar_fontsize,
        fontweight='bold',
        labelpad=15
    )
    cbar.ax.tick_params(
        labelsize=colorbar_fontsize
    )

    tp_count = len(edges['TP'])
    fp_count = len(edges['FP'])
    fn_count = len(edges['FN'])

    legend = [
        Line2D(
            [0], [0],
            color=CMAP_TP(0.75),
            lw=2.8,
            alpha=0.9,
            label=f'Correct (TP, {tp_count})'
        ),
        Line2D(
            [0], [0],
            color=COLOR_FP,
            lw=2.8,
            label=f'Spurious (FP, {fp_count})'
        ),
        Line2D(
            [0], [0],
            color=COLOR_FN,
            lw=2.8,
            label=f'Missed (FN, {fn_count})'
        ),
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor=NODE_COLOR,
            markeredgecolor='white',
            markersize=10,
            label='Node'
        )
    ]

    fig.legend(
        handles=legend,
        loc='lower center',
        bbox_to_anchor=(0.46, 0.022),
        ncol=4,
        frameon=False,
        fontsize=legend_fontsize,
        columnspacing=2.4,
        handlelength=2.4
    )

    fig.savefig(
        OUTPUT_FILE,
        format='pdf',
        dpi=600,
        facecolor='white',
        bbox_inches='tight'
    )

    print(f'Saved figure: {OUTPUT_FILE}')
    plt.show()


# ============================================================
# 2. Reproduce the two original M=700 variants
# ============================================================
# Original standard-font version
plot_large_lorenz_xy_network(
    base_fontsize=20,
    node_size=24,
    colorbar_fontsize=25,
    legend_fontsize=25,
    close_before=True
)
